In [1]:
import os
import glob
import re
import pandas as pd

In [2]:
ROGUE_DIR = "rogue_purity_results"
OUT_DIR   = "rogue_purity_results"

In [3]:
def summarize_rogue_per_gep(csv_path):
    df = pd.read_csv(csv_path)
    summary = (df.groupby('cluster')
                 .agg(n_samples   =('sample',  'nunique'),
                      median_ROGUE=('ROGUE',   'median'),
                      mean_ROGUE  =('ROGUE',   'mean'),
                      min_ROGUE   =('ROGUE',   'min'),
                      max_ROGUE   =('ROGUE',   'max'),
                      total_cells =('n_cells', 'sum'))
                 .round(3)
                 .rename_axis('GEP')
                 .reset_index())
    return summary

In [4]:
# Discover all per-K CSVs
files = sorted(glob.glob(os.path.join(ROGUE_DIR, "rogue_per_gep_K*.csv")),
               key=lambda f: int(re.search(r'K(\d+)', f).group(1)))

In [5]:
all_summaries = []
for f in files:
    k = int(re.search(r'K(\d+)', os.path.basename(f)).group(1))
    summary = summarize_rogue_per_gep(f)
    summary.insert(0, 'K', k)

    # Per-K summary CSV (drops the K column for the side-by-side view)
    out_path = os.path.join(OUT_DIR, f"rogue_per_gep_K{k}_summary.csv")
    summary.drop(columns='K').to_csv(out_path, index=False)
    print(f"\nK={k}  →  {out_path}")
    print(summary.drop(columns='K').to_string(index=False))

    all_summaries.append(summary)


K=3  →  rogue_purity_results/rogue_per_gep_K3_summary.csv
 GEP  n_samples  median_ROGUE  mean_ROGUE  min_ROGUE  max_ROGUE  total_cells
   1         56         0.754       0.754      0.623      0.855        18670
   2         27         0.510       0.534      0.391      0.841         6412
   3         30         0.844       0.834      0.601      0.973         1048

K=4  →  rogue_purity_results/rogue_per_gep_K4_summary.csv
 GEP  n_samples  median_ROGUE  mean_ROGUE  min_ROGUE  max_ROGUE  total_cells
   1         56         0.763       0.761      0.640      0.874        18689
   2         28         0.527       0.541      0.398      0.779         5302
   3         29         0.880       0.842      0.642      0.973         1016
   4         13         0.708       0.725      0.546      0.870         1077

K=5  →  rogue_purity_results/rogue_per_gep_K5_summary.csv
 GEP  n_samples  median_ROGUE  mean_ROGUE  min_ROGUE  max_ROGUE  total_cells
   1         56         0.763       0.758      0.629 

In [6]:
# Combined long-format table across all K (handy for §5a and downstream plots)
combined = pd.concat(all_summaries, ignore_index=True)
combined_path = os.path.join(OUT_DIR, "rogue_per_gep_summary_allK.csv")
combined.to_csv(combined_path, index=False)
print(f"\nCombined summary → {combined_path}  ({len(combined)} rows)")


Combined summary → rogue_purity_results/rogue_per_gep_summary_allK.csv  (96 rows)
